In [ ]:
import numpy as np
import pandas as pd
import json 
import ast


In [ ]:
df = pd.read_csv('SGJobData_updated.csv')
print(df.info())

FileNotFoundError: [Errno 2] No such file or directory: 'SGJobData_updated.csv'

In [ ]:
#df.iloc[0:50000].to_excel('first 50k.xlsx')

# Cleaning up steps #
1) Remove truly blank roles
3) Remove duplicates if any.

# Remove truly blank rows #

In [ ]:
# Remove truly blank rows #

categories_blank = df['categories'].isna() | (df['categories'].str.strip() == '')
employmentTypes_blank = df['employmentTypes'].isna() | (df['employmentTypes'].str.strip() == '')

print("categories blank count:", categories_blank.sum())
print("employmentTypes blank count:", employmentTypes_blank.sum())
print("both blank:", (categories_blank & employmentTypes_blank).sum())
print("categories blank but employmentTypes not:", (categories_blank & ~employmentTypes_blank).sum())
print("employmentTypes blank but categories not:", (~categories_blank & employmentTypes_blank).sum())
print("same rows (masks identical):", categories_blank.equals(employmentTypes_blank))


categories blank count: 0
employmentTypes blank count: 0
both blank: 0
categories blank but employmentTypes not: 0
employmentTypes blank but categories not: 0
same rows (masks identical): True


# Remove empty rows and empty columns

In [ ]:
df1 = df[df['categories'].notna() & (df['categories'].str.strip() != '')].drop(columns=['occupationId', 'status_id'])
print(df1.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1044597 entries, 0 to 1044596
Data columns (total 22 columns):
 #   Column                              Non-Null Count    Dtype  
---  ------                              --------------    -----  
 0   categories                          1044597 non-null  object 
 1   employmentTypes                     1044597 non-null  object 
 2   metadata_expiryDate                 1044597 non-null  object 
 3   metadata_isPostedOnBehalf           1044597 non-null  bool   
 4   metadata_jobPostId                  1044597 non-null  object 
 5   metadata_newPostingDate             1044597 non-null  object 
 6   metadata_originalPostingDate        1044597 non-null  object 
 7   metadata_repostCount                1044597 non-null  int64  
 8   metadata_totalNumberJobApplication  1044597 non-null  int64  
 9   metadata_totalNumberOfView          1044597 non-null  int64  
 10  minimumYearsExperience              1044597 non-null  int64  
 11  numberOfVac

# Change title to all lower case #

# Change data type to the right ones and change title to lower case #

In [ ]:
date_cols = ['metadata_expiryDate', 'metadata_newPostingDate', 'metadata_originalPostingDate']
category_cols = ['employmentTypes', 'positionLevels', 'postedCompany_name', 'salary_type', 'status_jobStatus']

df1[date_cols] = df1[date_cols].apply(pd.to_datetime)
df1[category_cols] = df1[category_cols].astype('category')

df1['metadata_jobPostId'] = df1['metadata_jobPostId'].astype('string')
df1['title'] = df1['title'].astype('string').str.lower()

print(df1.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1044597 entries, 0 to 1044596
Data columns (total 22 columns):
 #   Column                              Non-Null Count    Dtype         
---  ------                              --------------    -----         
 0   categories                          1044597 non-null  object        
 1   employmentTypes                     1044597 non-null  category      
 2   metadata_expiryDate                 1044597 non-null  datetime64[ns]
 3   metadata_isPostedOnBehalf           1044597 non-null  bool          
 4   metadata_jobPostId                  1044597 non-null  string        
 5   metadata_newPostingDate             1044597 non-null  datetime64[ns]
 6   metadata_originalPostingDate        1044597 non-null  datetime64[ns]
 7   metadata_repostCount                1044597 non-null  int64         
 8   metadata_totalNumberJobApplication  1044597 non-null  int64         
 9   metadata_totalNumberOfView          1044597 non-null  int64         

In [ ]:
# ============================================================
# CLEAN AND AUDIT WHITESPACE IN TEXT COLUMNS
# ============================================================

# Columns to clean
clean_cols = ["title", "postedCompany_name"]

# Store original values before cleaning
original_values = {
    col: df[col].copy()
    for col in clean_cols
}

# ------------------------------------------------------------
# 1. Check whitespace problems before cleaning
# ------------------------------------------------------------
print("WHITESPACE ISSUES BEFORE CLEANING")
print("=" * 50)

for col in clean_cols:
    series = df[col].astype("string")

    leading_count = series.str.startswith(" ", na=False).sum()
    trailing_count = series.str.endswith(" ", na=False).sum()
    multiple_count = series.str.contains(r"\s{2,}", regex=True, na=False).sum()

    print(f"\nColumn: {col}")
    print(f"Leading spaces : {leading_count}")
    print(f"Trailing spaces: {trailing_count}")
    print(f"Multiple spaces: {multiple_count}")


# ------------------------------------------------------------
# 2. Clean leading, trailing and multiple spaces
# ------------------------------------------------------------
for col in clean_cols:
    df[col] = (
        df[col]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )


# ------------------------------------------------------------
# 3. Identify rows changed by cleaning
# ------------------------------------------------------------
title_changed = (
    original_values["title"].astype("string")
    != df["title"]
)

company_changed = (
    original_values["postedCompany_name"].astype("string")
    != df["postedCompany_name"]
)

print("\nROWS CHANGED")
print("=" * 50)
print(f"Titles changed      : {title_changed.sum()}")
print(f"Company names changed: {company_changed.sum()}")


# ------------------------------------------------------------
# 4. Display examples before and after cleaning
# ------------------------------------------------------------
title_changes = pd.DataFrame({
    "metadata_jobPostId": df.loc[title_changed, "metadata_jobPostId"],
    "original_title": original_values["title"].loc[title_changed],
    "cleaned_title": df.loc[title_changed, "title"]
})

company_changes = pd.DataFrame({
    "metadata_jobPostId": df.loc[company_changed, "metadata_jobPostId"],
    "original_company_name": original_values["postedCompany_name"].loc[company_changed],
    "cleaned_company_name": df.loc[company_changed, "postedCompany_name"]
})

print("\nTITLE CHANGES")
print("=" * 50)
print(title_changes.head(20).to_string(index=False))

print("\nCOMPANY NAME CHANGES")
print("=" * 50)
print(company_changes.head(20).to_string(index=False))


# ------------------------------------------------------------
# 5. Verify whitespace issues after cleaning
# ------------------------------------------------------------
print("\nWHITESPACE ISSUES AFTER CLEANING")
print("=" * 50)

remaining_masks = {}

for col in clean_cols:
    series = df[col].astype("string")

    remaining_mask = (
        series.str.startswith(" ", na=False)
        | series.str.endswith(" ", na=False)
        | series.str.contains(r"\s{2,}", regex=True, na=False)
    )

    remaining_masks[col] = remaining_mask

    leading_count = series.str.startswith(" ", na=False).sum()
    trailing_count = series.str.endswith(" ", na=False).sum()
    multiple_count = series.str.contains(r"\s{2,}", regex=True, na=False).sum()

    print(f"\nColumn: {col}")
    print(f"Leading spaces remaining : {leading_count}")
    print(f"Trailing spaces remaining: {trailing_count}")
    print(f"Multiple spaces remaining: {multiple_count}")


# ------------------------------------------------------------
# 6. Show any records that still contain whitespace problems
# ------------------------------------------------------------
title_remaining = remaining_masks["title"]
company_remaining = remaining_masks["postedCompany_name"]

if title_remaining.any():
    print("\nTITLE RECORDS STILL CONTAINING WHITESPACE ISSUES")
    print(
        df.loc[
            title_remaining,
            ["metadata_jobPostId", "title"]
        ].head(20).to_string(index=False)
    )
else:
    print("\nNo remaining whitespace problems in title.")


if company_remaining.any():
    print("\nCOMPANY RECORDS STILL CONTAINING WHITESPACE ISSUES")
    print(
        df.loc[
            company_remaining,
            ["metadata_jobPostId", "postedCompany_name"]
        ].head(20).to_string(index=False)
    )
else:
    print("No remaining whitespace problems in postedCompany_name.")


# ------------------------------------------------------------
# 7. Optional: save cleaned dataset
# ------------------------------------------------------------
output_file = "SGJobData_whitespace_cleaned.csv"

df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print(f"\nCleaned file saved as: {output_file}")

            

WHITESPACE ISSUES BEFORE CLEANING

Column: title
Leading spaces : 0
Trailing spaces: 0
Multiple spaces: 0

Column: postedCompany_name
Leading spaces : 0
Trailing spaces: 0
Multiple spaces: 0

ROWS CHANGED
Titles changed      : 0
Company names changed: 0

TITLE CHANGES
Empty DataFrame
Columns: [metadata_jobPostId, original_title, cleaned_title]
Index: []

COMPANY NAME CHANGES
Empty DataFrame
Columns: [metadata_jobPostId, original_company_name, cleaned_company_name]
Index: []

WHITESPACE ISSUES AFTER CLEANING

Column: title
Leading spaces remaining : 0
Trailing spaces remaining: 0
Multiple spaces remaining: 0

Column: postedCompany_name
Leading spaces remaining : 0
Trailing spaces remaining: 0
Multiple spaces remaining: 0

No remaining whitespace problems in title.
No remaining whitespace problems in postedCompany_name.

Cleaned file saved as: SGJobData_whitespace_cleaned.csv


In [ ]:
# Remove the "PTE. LTD." and "LTD." from the postedCompany_name column and convert to uppercase
df["postedCompany_name"] = (
    df["postedCompany_name"]
    .str.replace(".", "", regex=False)
    .str.upper()
)
df["postedCompany_name"] = (
    df["postedCompany_name"]
    .str.replace(".", "", regex=False)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .str.upper()
)
df[df["postedCompany_name"].str.contains("LTD", case=False, na=False)]["postedCompany_name"].head(20)


0                  WORKSTONE PTE LTD
1              TRUST RECRUIT PTE LTD
2           PU TIEN SERVICES PTE LTD
3              TRUST RECRUIT PTE LTD
4     EATZ CATERING SERVICES PTE LTD
5     BYTECENTURE CONSULTING PTE LTD
6              TRUST RECRUIT PTE LTD
7                  TRITON AI PTE LTD
8                  WORKSTONE PTE LTD
9                  WORKSTONE PTE LTD
10                 WORKSTONE PTE LTD
11                 SAVORNANA PTE LTD
12                 WORKSTONE PTE LTD
13             TRUST RECRUIT PTE LTD
14             SDT MOLECULAR PTE LTD
18                 WORKSTONE PTE LTD
19                 WORKSTONE PTE LTD
20             TRUST RECRUIT PTE LTD
21             TRUST RECRUIT PTE LTD
22                 TRITON AI PTE LTD
Name: postedCompany_name, dtype: string

In [ ]:
df["salary_average"] = (
    df["salary_minimum"] +
    df["salary_maximum"]
) / 2
salary_anomaly = df[
    (df["salary_minimum"] < 500) |
    (df["salary_maximum"] > 50000) |
    (df["salary_minimum"] > df["salary_maximum"]) |
    (df["salary_maximum"] >= df["salary_minimum"] * 10)
]

print("Salary anomalies:", len(salary_anomaly))

salary_anomaly.head()
vacancy_anomaly = df[
    (df["numberOfVacancies"] <= 0) |
    (df["numberOfVacancies"] > 100)
]

print("Vacancy anomalies:", len(vacancy_anomaly))

vacancy_anomaly.head()
experience_anomaly = df[
    (df["minimumYearsExperience"] < 0) |
    (df["minimumYearsExperience"] > 40)
]

print("Experience anomalies:", len(experience_anomaly))

experience_anomaly.head()
all_anomalies = pd.concat(
    [
        salary_anomaly,
        vacancy_anomaly,
        experience_anomaly
    ]
).drop_duplicates()

print("Total unique anomalies:", len(all_anomalies))
all_anomalies.to_csv(
    "Job_Anomalies.csv",
    index=False
)

print("File saved successfully.")
all_anomalies[
    [
        "metadata_jobPostId",
        "postedCompany_name",
        "title",
        "salary_minimum",
        "salary_maximum",
        "numberOfVacancies",
        "minimumYearsExperience"
    ]
].head(20)



Salary anomalies: 10328
Vacancy anomalies: 308
Experience anomalies: 21
Total unique anomalies: 10605
File saved successfully.


,metadata_jobPostId,postedCompany_name,title,salary_minimum,salary_maximum,numberOfVacancies,minimumYearsExperience
167,MCF-2023-0065764,OLIVE TREE INTERVENTION CENTRE PTE LTD,Educational Therapist - Internship,1,1,3,0
455,MCF-2023-0040383,I-CONSULT TECH PTE LTD,Part-Time Project / Sales Admin Officer,8,10,1,1
512,MCF-2023-0274391,METRO (PRIVATE) LIMITED,Replenishment Crew (Part Time),9,10,2,0
619,MCF-2023-0274482,BRAWN & BRAINS COFFEE PTE LTD,Kitchen Assistant (Part Time),12,15,3,2
639,MCF-2023-0169711,LS CREATIVO MARKETING (S) PTE LTD,Part time Corporate sales specialist,480,1280,2,1
695,MCF-2023-0166414,BRAWN & BRAINS COFFEE PTE LTD,Part Time Dishwasher,12,15,2,1
703,MCF-2023-0274539,CHINESE DEVELOPMENT ASSISTANCE COUNCIL,Executive or Senior Executive (Family and Work...,1,1,2,1
1199,MCF-2023-0273943,ITUITION RENTAL,Part Time or Full Time Private Tuition Tutor,240,6000,5,0
1681,MCF-2023-0271968,BGC GROUP PTE LTD,[GOVT] Project Coordinator (Healthcare) | $10....,9,11,1,0
1978,MCF-2023-0270547,SUCCESS HUMAN RESOURCE CENTRE PTE LTD,Temporary Beauty Stylist,10,12,5,1


In [ ]:
# Add the original CSV row number
# +2 because CSV row 1 is the header
df["csv_row_number"] = df.index + 2

# Convert salary and repost-count columns to numeric
numeric_columns = [
    "salary_minimum",
    "salary_maximum",
    "average_salary",
    "metadata_repostCount"
]

for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

# Convert posting-date columns to dates
date_columns = [
    "metadata_originalPostingDate",
    "metadata_newPostingDate",
    "metadata_expiryDate"
]

for column in date_columns:
    df[column] = pd.to_datetime(df[column], errors="coerce")
salary_issues = []

expected_average = (
    df["salary_minimum"] + df["salary_maximum"]
) / 2

salary_rules = {
    "Missing minimum salary":
        df["salary_minimum"].isna(),

    "Missing maximum salary":
        df["salary_maximum"].isna(),

    "Missing average salary":
        df["average_salary"].isna(),

    "Minimum salary is negative":
        df["salary_minimum"] < 0,

    "Maximum salary is negative":
        df["salary_maximum"] < 0,

    "Minimum salary is zero":
        df["salary_minimum"] == 0,

    "Maximum salary is zero":
        df["salary_maximum"] == 0,

    "Minimum salary exceeds maximum salary":
        df["salary_minimum"] > df["salary_maximum"],

    "Average salary is below minimum salary":
        df["average_salary"] < df["salary_minimum"],

    "Average salary exceeds maximum salary":
        df["average_salary"] > df["salary_maximum"],

    "Average salary calculation is inconsistent":
        (
            df["average_salary"].notna()
            & expected_average.notna()
            & ((df["average_salary"] - expected_average).abs() > 0.01)
        ),

    "Monthly minimum salary below 500":
        (
            df["salary_type"].eq("Monthly")
            & (df["salary_minimum"] < 500)
        ),

    "Monthly maximum salary above 50000":
        (
            df["salary_type"].eq("Monthly")
            & (df["salary_maximum"] > 50000)
        ),

    "Maximum salary is at least 10 times minimum salary":
        (
            (df["salary_minimum"] > 0)
            & (
                df["salary_maximum"]
                >= df["salary_minimum"] * 10
            )
        )
}

for issue_name, condition in salary_rules.items():
    result = df.loc[
        condition,
        [
            "csv_row_number",
            "metadata_jobPostId",
            "postedCompany_name",
            "title",
            "salary_minimum",
            "salary_maximum",
            "average_salary",
            "salary_type"
        ]
    ].copy()

    result["audit_area"] = "Salary"
    result["issue"] = issue_name
    salary_issues.append(result)

salary_inconsistencies = pd.concat(
    salary_issues,
    ignore_index=True
)

print("Salary issue flags:", len(salary_inconsistencies))
salary_inconsistencies.head(20)
posting_date_issues = []

date_rules = {
    "Invalid or missing original posting date":
        df["metadata_originalPostingDate"].isna(),

    "Invalid or missing new posting date":
        df["metadata_newPostingDate"].isna(),

    "Invalid or missing expiry date":
        df["metadata_expiryDate"].isna(),

    "New posting date is before original posting date":
        (
            df["metadata_newPostingDate"]
            < df["metadata_originalPostingDate"]
        ),

    "Expiry date is before original posting date":
        (
            df["metadata_expiryDate"]
            < df["metadata_originalPostingDate"]
        ),

    "Expiry date is before new posting date":
        (
            df["metadata_expiryDate"]
            < df["metadata_newPostingDate"]
        ),

    "Repost count is negative":
        df["metadata_repostCount"] < 0,

    "Repost count is zero but dates are different":
        (
            df["metadata_repostCount"].eq(0)
            & df["metadata_originalPostingDate"].notna()
            & df["metadata_newPostingDate"].notna()
            & (
                df["metadata_originalPostingDate"]
                != df["metadata_newPostingDate"]
            )
        ),

    "Repost count is positive but dates are identical":
        (
            df["metadata_repostCount"].gt(0)
            & df["metadata_originalPostingDate"].notna()
            & df["metadata_newPostingDate"].notna()
            & (
                df["metadata_originalPostingDate"]
                == df["metadata_newPostingDate"]
            )
        )
}

for issue_name, condition in date_rules.items():
    result = df.loc[
        condition,
        [
            "csv_row_number",
            "metadata_jobPostId",
            "postedCompany_name",
            "title",
            "metadata_originalPostingDate",
            "metadata_newPostingDate",
            "metadata_expiryDate",
            "metadata_repostCount"
        ]
    ].copy()

    result["audit_area"] = "Posting Date"
    result["issue"] = issue_name
    posting_date_issues.append(result)

posting_date_inconsistencies = pd.concat(
    posting_date_issues,
    ignore_index=True
)

print(
    "Posting-date issue flags:",
    len(posting_date_inconsistencies)
)

posting_date_inconsistencies.head(20)
def parse_categories(value):
    """Convert the categories JSON text into ID and name lists."""
    try:
        records = json.loads(value)

        ids = [
            str(item.get("id")).strip()
            for item in records
            if item.get("id") is not None
        ]

        names = [
            str(item.get("category")).strip()
            for item in records
            if item.get("category") is not None
        ]

        return ids, names, None

    except (TypeError, json.JSONDecodeError):
        return [], [], "Invalid categories JSON"


def split_values(value):
    """Split comma-separated Category_ID or Category_Name values."""
    if pd.isna(value):
        return []

    return [
        item.strip()
        for item in str(value).split(",")
        if item.strip()
    ]
category_issue_records = []

for index, row in df.iterrows():

    json_ids, json_names, json_error = parse_categories(
        row["categories"]
    )

    listed_ids = split_values(row["Category_ID"])
    listed_names = split_values(row["Category_Name"])

    issues = []

    if json_error:
        issues.append(json_error)

    if not json_error:
        if json_ids != listed_ids:
            issues.append(
                "Category_ID does not match categories JSON"
            )

        if json_names != listed_names:
            issues.append(
                "Category_Name does not match categories JSON"
            )

        if len(json_ids) != len(json_names):
            issues.append(
                "JSON category ID and name counts differ"
            )

    if len(listed_ids) != len(listed_names):
        issues.append(
            "Category_ID and Category_Name counts differ"
        )

    if len(listed_ids) != len(set(listed_ids)):
        issues.append(
            "Duplicate Category_ID within the same row"
        )

    if len(listed_names) != len(set(listed_names)):
        issues.append(
            "Duplicate Category_Name within the same row"
        )

    for issue in issues:
        category_issue_records.append({
            "csv_row_number": row["csv_row_number"],
            "metadata_jobPostId": row["metadata_jobPostId"],
            "postedCompany_name": row["postedCompany_name"],
            "title": row["title"],
            "categories": row["categories"],
            "Category_ID": row["Category_ID"],
            "Category_Name": row["Category_Name"],
            "audit_area": "Category",
            "issue": issue
        })

category_inconsistencies = pd.DataFrame(
    category_issue_records
)

print(
    "Category issue flags:",
    len(category_inconsistencies)
)

category_inconsistencies.head(20)
all_inconsistencies = pd.concat(
    [
        salary_inconsistencies,
        posting_date_inconsistencies,
        category_inconsistencies
    ],
    ignore_index=True,
    sort=False
)

all_inconsistencies = all_inconsistencies.sort_values(
    by=["audit_area", "issue", "csv_row_number"]
)

print("Total inconsistency flags:", len(all_inconsistencies))

all_inconsistencies.head(50)
issue_summary = (
    all_inconsistencies
    .groupby(["audit_area", "issue"])
    .size()
    .reset_index(name="number_of_flags")
    .sort_values(
        by=["audit_area", "number_of_flags"],
        ascending=[True, False]
    )
)

issue_summary
all_inconsistencies.to_csv(
    "Salary_Date_Category_Inconsistencies.csv",
    index=False
)

issue_summary.to_csv(
    "Salary_Date_Category_Issue_Summary.csv",
    index=False
)

print("Files saved:")
print("1. Salary_Date_Category_Inconsistencies.csv")
print("2. Salary_Date_Category_Issue_Summary.csv")

category_inconsistencies


Salary issue flags: 11848
Posting-date issue flags: 88
Category issue flags: 0
Total inconsistency flags: 11936
Files saved:
1. Salary_Date_Category_Inconsistencies.csv
2. Salary_Date_Category_Issue_Summary.csv


""


In [ ]:
# Columns to check for duplicates (all except metadata_jobPostId)
cols_to_check = [col for col in df1.columns if col != "metadata_jobPostId"]

# Find rows that are duplicated based on those columns
# keep=False marks ALL occurrences (not just the 2nd, 3rd, etc.) as duplicates
duplicate_mask = df1.duplicated(subset=cols_to_check, keep=False)

duplicate_rows = df1[duplicate_mask]

# Optional: sort so duplicate groups sit next to each other for easy comparison
duplicate_rows = duplicate_rows.sort_values(by=cols_to_check)

print(f"Found {len(duplicate_rows)} duplicate rows")
# Drop duplicates, keeping the first occurrence of each group
df2 = df1.drop_duplicates(subset=cols_to_check, keep='first')
#print(f"Remaining rows after dropping duplicates: {len(df2)}")
df2.info()

Found 19709 duplicate rows
<class 'pandas.core.frame.DataFrame'>
Index: 1031725 entries, 0 to 1044596
Data columns (total 22 columns):
 #   Column                              Non-Null Count    Dtype         
---  ------                              --------------    -----         
 0   categories                          1031725 non-null  object        
 1   employmentTypes                     1031725 non-null  category      
 2   metadata_expiryDate                 1031725 non-null  datetime64[ns]
 3   metadata_isPostedOnBehalf           1031725 non-null  bool          
 4   metadata_jobPostId                  1031725 non-null  string        
 5   metadata_newPostingDate             1031725 non-null  datetime64[ns]
 6   metadata_originalPostingDate        1031725 non-null  datetime64[ns]
 7   metadata_repostCount                1031725 non-null  int64         
 8   metadata_totalNumberJobApplication  1031725 non-null  int64         
 9   metadata_totalNumberOfView          1031725 no

In [ ]:
df2.to_csv('SGJobData_cleaned.csv', index=False)

# Not necessary since Streamlit can process columns with json data well? # Stop here for now

import json



df1['categories_parsed'] = df1['categories'].apply(json.loads)
df1['categories_parsed'].head().to_excel('categories_parsed check.xlsx')


df_categories = df1[['metadata_jobPostId', 'categories_parsed']].explode('categories_parsed')
df_categories = df_categories.dropna(subset=['categories_parsed'])
df_categories['category_id'] = df_categories['categories_parsed'].apply(lambda d: d['id'])
df_categories['category'] = df_categories['categories_parsed'].apply(lambda d: d['category'])
df_categories = df_categories.drop(columns='categories_parsed')

df_categories.head()

def parse_json(val):
    if isinstance(val, str):
        return json.loads(val)
    return val

records = df1['categories'].apply(parse_json).explode()
result_df = pd.json_normalize(records)[['id', 'category']]
result_df = result_df.drop_duplicates(subset='id').reset_index(drop=True)
result_df.head()

result_df = result_df.drop_duplicates(subset='id').reset_index(drop=True)
result_df



if isinstance(df1['categories'].iloc[0], str):
    df1['categories'] = df1['categories'].apply(ast.literal_eval)

# Step 2: explode so each category dict gets its own row
df_map = df1[['metadata_jobPostId', 'categories']].explode('categories').reset_index(drop=True)

# Step 3: pull out the 'id' (and optionally 'category' name) from each dict
df_map['category_id'] = df_map['categories'].apply(lambda x: x['id'])
df_map['category_name'] = df_map['categories'].apply(lambda x: x['category'])

# Step 4: drop the now-unneeded dict column
df_map = df_map.drop(columns=['categories'])
df_map